In [1]:
"""
stage_5.py -- Stage 5 (iteration 2): 5c2-raw hazard model, manifest-driven.

Frozen model contract (5c2-raw): grammar (bsince + ewm{2,6,18} per class, age, tod)
+ causal z-scored values @ {t-1, t-2, t-3}: signed & magnitude of every stream's
source column, leg amplitude, raw body signed & magnitude. Sign convention
confirming-positive (value * leg_dir).

Manifest-driven (iteration 2):
  - Fork axes (frame, session window, stream set) are READ from the Stage-0
    manifest, never redeclared here. Dropping a stream in Stage 0 (e.g. no-TICK)
    propagates automatically: grammar classes AND z-value channels both track the
    manifest stream set.
  - tod is derived from clock time (session_start), resolution-independent.
  - The manifest is baked into the model bundle so the booster carries its own
    contract; the worker asserts against it and prints it at startup.

Naming (iteration 2):
  SOURCE_PATH / src : the "source" oscillator file (HA OHLC + JMA + TICK + derivs),
                      Stage-0's input; rawer than bars/events. src is the primary
                      data frame, augmented in-place with raw body columns.
"""

import json
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
from scipy.signal import lfilter
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import log_loss, roc_auc_score
from datetime import datetime

from common import Featurizer, load_manifest, _expanding_z, _welford_check

In [2]:
import sys
from contextlib import contextmanager

class Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, text):
        for stream in self.streams: stream.write(text)
        return len(text)
    def flush(self):
        for stream in self.streams: stream.flush()
    def isatty(self): return False

@contextmanager
def copy_output_to(path, mode="a"):
    original_stdout, original_stderr = sys.stdout, sys.stderr

    with open(path, mode, encoding="utf-8", buffering=1) as log:
        sys.stdout = Tee(original_stdout, log)
        sys.stderr = Tee(original_stderr, log)
        try:
            yield
        finally:
            sys.stdout, sys.stderr = original_stdout, original_stderr

In [3]:
pd.set_option("display.width", 400)      # total characters per line
pd.set_option("display.max_columns", 30) # prevent wrapping by limiting columns
pd.set_option("display.max_rows", 1000)

In [4]:
# ---------------------------------------------------------------- CONFIG (per-run, explicit)
FRAME = 3
FRAME_TICK = 6
STAGE0_TAG = 'mnq-6T-3S-9-12am-RSX-vol2-v1'

MANIFEST_PATH = f"stage-0/{STAGE0_TAG}_manifest.json"
BARS_PATH = f"stage-0/{STAGE0_TAG}_bars.pqt"
EVENTS_PATH = f"stage-0/{STAGE0_TAG}_events.pqt"

ITER_DIR = "."                                   # iteration-2 root (encapsulated)
OUT_DIR = "stage-5"

VALID_FROM = "2025-07-01"
TRAIN_END = "2025-12-31"
TEST_FROM = "2026-01-01"

# frozen 5c2 architecture constants (NOT fork axes -- stay in code)
TAUS = (2.0, 6.0, 18.0)
VALUE_LAGS = (1, 2)                               # t-2, t-3 (t-1 shift is implicit)
ZWARM = 20
TOD_BIN_MIN = 30
BODY_OPEN_COL = "rawOpen"
BODY_CLOSE_COL = "rawLast"

LGBM_PARAMS = dict(
    objective="binary", metric="binary_logloss", learning_rate=0.05,
    num_leaves=127, min_data_in_leaf=1000, feature_fraction=0.9,
    bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
    num_threads=16, verbosity=-1,
)
NUM_ROUNDS = 8000
EARLY_STOP = 200
STAGE5_ANCHOR = 0.27402

LOG_FILE = f"logs/stage5-{STAGE0_TAG}-{datetime.now().strftime('%Y-%m-%d_%H-%M')}.txt"
# ----------------------------------------------------------------

In [5]:
# ---------------------------------------------------------------- grammar features
def build_grammar_features(fz, date_from=None, date_to=None):
    blocks = []
    for S in fz._selected(date_from, date_to):
        t = np.nonzero(~S["warm"])[0]
        n = S["n"]
        cols = []
        for c in fz.classes:
            P = S["P"][c]
            ind = np.diff(P).astype(np.float64)
            occ = np.where(ind > 0, np.arange(n), -1)
            last = np.maximum.accumulate(occ)
            lastm1 = np.concatenate(([-1], last[:-1]))
            bsince = np.where(lastm1 >= 0, np.arange(n) - lastm1, np.arange(n) + 1)
            cols.append(bsince[t])
            x = np.concatenate(([0.0], ind[:-1]))
            for tau in TAUS:
                a = np.exp(-1.0 / tau)
                s = lfilter([a], [1.0, -a], x)
                cols.append(s[t])
        lt = np.where(t > 0, S["lt_incl"][np.maximum(t - 1, 0)], -1)
        age = np.where(lt >= 0, t - lt, t + 1)
        cols.append(age)
        cols.append(S["tod"][t].astype(np.float64))
        blocks.append(np.stack(cols, 1).astype(np.float32))
    return np.concatenate(blocks), fz.grammar_names

In [6]:
# ---------------------------------------------------------------- value features
def value_base_names(manifest):
    """Value channels derived from the manifest stream set (source columns)."""
    cols = manifest["_stream_cols"]
    names = ([f"z_{c}_signed" for c in cols]
             + [f"z_{c}_mag" for c in cols]
             + ["z_leg_amp", f"z_body_raw_signed", f"z_body_raw_mag"])
    return cols, names


def build_value_features(fz, src, date_from=None, date_to=None):
    cols, base = value_base_names(fz.manifest)
    names = base + [f"{nm}_lag{L}" for L in VALUE_LAGS for nm in base]
    sv = src.set_index("timestamp")
    blocks = []
    for S in fz._selected(date_from, date_to):
        ts = pd.DatetimeIndex(S["timestamp"])
        r = sv.reindex(ts)
        leg_dir = S["leg_dir"]
        feats = []
        for c in cols:                                             # signed per stream col
            feats.append(_expanding_z(r[c].to_numpy(np.float64) * leg_dir, ZWARM))
        for c in cols:                                             # magnitude per stream col
            feats.append(_expanding_z(np.abs(r[c].to_numpy(np.float64)), ZWARM))
        feats.append(_expanding_z(np.abs(r["JMA"].to_numpy(np.float64) - S["leg_start_jma"]), ZWARM))
        bo = r[BODY_OPEN_COL].to_numpy(np.float64)
        bc = r[BODY_CLOSE_COL].to_numpy(np.float64)
        feats.append(_expanding_z((bc - bo) * leg_dir, ZWARM))     # confirming-positive
        feats.append(_expanding_z(np.abs(bc - bo), ZWARM))

        M = np.stack(feats, 1)
        M = np.concatenate([np.zeros((1, M.shape[1])), M[:-1]], 0)          # t-1 shift
        lagged = [M]
        for L in VALUE_LAGS:
            lagged.append(np.concatenate([np.zeros((L, M.shape[1])), M[:-L]], 0))
        M = np.concatenate(lagged, 1)
        t = np.nonzero(~S["warm"])[0]
        Mt = M[t]
        Mt[(t < ZWARM + max(VALUE_LAGS))] = 0.0
        blocks.append(Mt.astype(np.float32))
    return np.concatenate(blocks), names


def build_X(fz, src, date_from=None, date_to=None):
    Xg, gn = build_grammar_features(fz, date_from, date_to)
    Xv, vn = build_value_features(fz, src, date_from, date_to)
    assert len(Xg) == len(Xv), (len(Xg), len(Xv))
    return np.hstack([Xg, Xv]), gn + vn


def build_meta(fz, date_from=None, date_to=None):
    bi, ts, tg, dt = [], [], [], []
    for S in fz._selected(date_from, date_to):
        t = np.nonzero(~S["warm"])[0]
        bi.append(S["bar_index"][t])
        ts.append(S["timestamp"][t])
        tg.append(S["tgt"][t])
        dt.append(np.full(len(t), str(S["sess"])))
    return pd.DataFrame({"bar_index": np.concatenate(bi),
                         "timestamp": np.concatenate(ts),
                         "is_target": np.concatenate(tg),
                         "date": np.concatenate(dt)})

In [7]:
# ---------------------------------------------------------------- train / eval
def train(fz, src, train_end, valid_from):
    X, names = build_X(fz, src, None, train_end)
    meta = build_meta(fz, None, train_end)
    y = meta["is_target"].to_numpy().astype(np.int8)
    va = (meta["date"] >= valid_from).to_numpy()
    tr = ~va
    dtr = lgb.Dataset(X[tr], label=y[tr], feature_name=names)
    dva = lgb.Dataset(X[va], label=y[va], reference=dtr)
    booster = lgb.train(LGBM_PARAMS, dtr, num_boost_round=NUM_ROUNDS,
                        valid_sets=[dva], valid_names=["valid"],
                        callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False),
                                   lgb.log_evaluation(200)])
    p_va = booster.predict(X[va], num_iteration=booster.best_iteration)
    iso = IsotonicRegression(out_of_bounds="clip").fit(p_va, y[va])
    print(json.dumps(dict(n_train=int(tr.sum()), n_valid=int(va.sum()),
                          best_iteration=int(booster.best_iteration),
                          valid_logloss_cal=float(log_loss(y[va], iso.predict(p_va)))),
                     indent=2))
    imp = pd.DataFrame({"feature": names,
                        "gain": booster.feature_importance("gain")}
                       ).sort_values("gain", ascending=False)
    print(imp.to_string(index=False))
    return dict(booster=booster, iso=iso, feature_names=names,
                valid_from=valid_from, train_end=train_end,
                manifest=fz.manifest, tag=STAGE0_TAG, importance=imp)


def evaluate(fz, src, model, start, end=None, anchor=STAGE5_ANCHOR):
    X, _ = build_X(fz, src, start, end)
    meta = build_meta(fz, start, end)
    y = meta["is_target"].to_numpy().astype(np.int8)
    p = model["booster"].predict(X, num_iteration=model["booster"].best_iteration)
    p_cal = model["iso"].predict(p)
    ll_cal = log_loss(y, p_cal)
    ll_const = log_loss(y, np.full_like(p, y.mean(), dtype=np.float64))
    print(json.dumps(dict(n_rows=int(len(y)), holdout_logloss_cal=float(ll_cal),
                          holdout_logloss_const=float(ll_const),
                          skill=float(1 - ll_cal / ll_const),
                          auc=float(roc_auc_score(y, p)),
                          anchor=anchor, delta=float(ll_cal - anchor)), indent=2))
    out = meta[["bar_index", "timestamp", "is_target"]].copy()
    out["p"] = p.astype(np.float32)
    out["p_cal"] = p_cal.astype(np.float32)
    tbl = out.assign(bin=pd.qcut(out["p_cal"], 10, duplicates="drop")).groupby(
        "bin", observed=True).agg(mean_p=("p_cal", "mean"),
                                  realized=("is_target", "mean"), n=("p_cal", "size"))
    print(tbl.to_string())
    return out

In [8]:
# ---------------------------------------------------------------- run
manifest = load_manifest(MANIFEST_PATH,TOD_BIN_MIN)

SOURCE_PATH = manifest["source_file"]

assert STAGE0_TAG == manifest["stage0_tag"]

bars = pd.read_parquet(BARS_PATH)
events = pd.read_parquet(EVENTS_PATH)

sess_lo = pd.Timestamp(manifest["session_start"]).time()
sess_hi = pd.Timestamp(manifest["session_end"]).time()

src = pd.read_parquet(SOURCE_PATH)
src = src[(src["timestamp"].dt.time >= sess_lo) & (src["timestamp"].dt.time < sess_hi)]

###  assert day start exists in src and raw ###
src_min_time = src["timestamp"].dt.time.min()
print(f'SRC MIN TIME: {src_min_time}')
print(f'dates range: {src["date"].min()} .. {src["date"].max()}')    

assert sess_lo == src_min_time

assert src[["rawOpen", "rawLast"]].notna().all().all(), "raw OHLC has gaps vs source timestamps"

SRC MIN TIME: 09:00:00
dates range: 2022-01-03 00:00:00 .. 2026-07-24 00:00:00


In [ ]:
with copy_output_to(LOG_FILE):
    print('----------------------------- !! VERIFY !! -----------------------------')
    print(f'FRAME: {FRAME}sec {FRAME_TICK}t, STAGE0_TAG: {STAGE0_TAG}')
    print('------------------------------- MANIFEST -------------------------------')
    print(json.dumps({k: v for k, v in manifest.items() if not k.startswith("_")}, indent=2))
    print('------------------------------------------------------------------------')
    
    #
    
    fz = Featurizer(bars, events, manifest, TOD_BIN_MIN, TAUS)
    #augment_featurizer(fz, bars)
    
    S0 = fz.sessions[len(fz.sessions) // 2]
    xchk = src.set_index("timestamp").reindex(pd.DatetimeIndex(S0["timestamp"]))["jmaD1"].to_numpy(np.float64)
    print("welford max abs diff:", _welford_check(xchk, ZWARM))
    
    model = train(fz, src, TRAIN_END, VALID_FROM)
    pred = evaluate(fz, src, model, TEST_FROM)
    
    print(f"-------------- {STAGE0_TAG} --------------")
    joblib_file = f"{OUT_DIR}/{STAGE0_TAG}_model.joblib"
    importance_file = f"{OUT_DIR}/{STAGE0_TAG}_importance.csv"
    pred_file = f"{OUT_DIR}/{STAGE0_TAG}_pred.pqt"
  
    joblib.dump({k: v for k, v in model.items() if k != "importance"}, joblib_file)
    model["importance"].to_csv(importance_file, index=False)
    pred.to_parquet(pred_file, index=False)
    
    print(f'    joblib_file: {joblib_file}')
    print(f'importance_file: {importance_file}')
    print(f'      pred_file: {pred_file}')
    
    #
    
    z = pred.timestamp >= TEST_FROM
    print("holdout-window logloss:", log_loss(pred.is_target[z], pred.p_cal[z]))
    
    #

In [24]:

print('\n--------------------------- SUMMARY ---------------------------\n')

p_date = pred['timestamp'].dt.normalize()

h = pred[p_date >= TEST_FROM]          # no-op if the parquet is holdout-only
GREEN = float(h["p_cal"].quantile(0.50))
RED   = float(h["p_cal"].quantile(0.90))
g = h[h["p_cal"] <  GREEN]
r = h[h["p_cal"] >= RED]
print(f"GREEN {GREEN:.8g} {len(g)/len(h):.2%} of bars wrong 1/{1/g['is_target'].mean():.0f}")
print(f"RED   {RED:.8g}   {len(r)/len(h):.2%} of bars right {r['is_target'].mean():.1%}")

# BETTER RED
print()

for q in [0.90, 0.92, 0.94, 0.95, 0.96, 0.98]:
    t = h.p_cal.quantile(q)
    r = h[h.p_cal >= t]
    print(f"q {q:>4.2f}  cut {t:.6f}  {len(r)/len(h):6.2%} of bars right {r.is_target.mean():5.1%} - catches {r.is_target.sum()/h.is_target.sum():5.1%} of ends")

print()
#

y = h["is_target"].values.astype(np.float64)
pc = np.clip(h["p_cal"].values.astype(np.float64), 1e-15, 1-1e-15)
a  = y.mean()
ll_model = -(y*np.log(pc) + (1-y)*np.log(1-pc)).mean()
ll_const = -(a*np.log(a) + (1-a)*np.log(1-a))
print(f"y.mean {a:.5f}  ll_model {ll_model:.5f}  ll_const {ll_const:.5f}  skill {1-ll_model/ll_const:.4f}  rows {len(h)}")
    


--------------------------- SUMMARY ---------------------------

GREEN 0.00054837449 49.93% of bars wrong 1/4301
RED   0.25874126   10.07% of bars right 65.9%

q 0.90  cut 0.258741  10.07% of bars right 65.9% - catches 85.2% of ends
q 0.92  cut 0.402974   8.20% of bars right 74.1% - catches 78.1% of ends
q 0.94  cut 0.570854   6.09% of bars right 83.8% - catches 65.6% of ends
q 0.95  cut 0.669659   5.23% of bars right 87.8% - catches 59.0% of ends
q 0.96  cut 0.768366   4.10% of bars right 92.5% - catches 48.7% of ends
q 0.98  cut 0.953781   2.05% of bars right 98.4% - catches 25.9% of ends

y.mean 0.07784  ll_model 0.08845  ll_const 0.27347  skill 0.6766  rows 516883


-------------
volume 0 - 2026-07-08

volume 2 - 2026-07-24

data 

v0 - baseline, rxs/vel(last)

v1 - JMA(10)->JMA(15)

|##| volume | data ver | streams | frames | holdout LL | skill | AUC | const | Description |
|---|---|---|---|---|---|---|---|---|---|
| 1 | 0 | 0 | 4 | 3/6s | 0.12311721078371671 | 0.6308157000533998 | 0.971575163079359 | 0.3334844163241089 | baseline - equals to iter3 (prod) |
| 2 | 2 | 0| 4 | 3/6s | 0.12255218585307534 | 0.6322527034711443 | 0.9718959058942918 | 0.3332510857587206 | baseline for volume 2 |
| 3 | 2 | 0| 4 RSX | 3/6s | 0.12098306134744605 | 0.6369612387848608 | 0.9725483537461633 | 0.3332510857587206 | RSX Last |
| 4 | 2 | 1| 4 RSX | 3/6s | 0.08845170818921866 | 0.6765587167323781 | 0.9805359082936125 | 0.2734706815890039 | RSX Last JMA15 |

  "holdout_logloss_cal": 0.08845170818921866,
  "holdout_logloss_const": 0.2734706815890039,
  "skill": 0.6765587167323781,
  "auc": 0.9805359082936125,


RED LAMP FINE TUNING
--------------------
The exchange rate degrades as you climb: 0.90→0.94 buys 18pp of precision for 20pp of recall, roughly even money. 0.96→0.98 costs 23pp of recall for 6pp of precision — the expensive end.

But even money isn't the criterion. For your use, recall is nearly free to give up: you've said missing an end is fine, and a missed end self-corrects anyway — the JMA slope flips and you can see it. A false RED costs more, because it spends attention and erodes trust in the lamp. So bias hard toward precision.

The stronger argument is that **the marker already carries the gradient.** Warming blue→yellow→orange is your "get ready." If the lamp is also an alert, it's redundant with the markers and noisier. Let the lamp be the *statement*: this is over. That points at q0.96 or q0.98, not q0.90.

Per session (~3000 bars in your 09:30–12:00 window, ~271 segments/day):

| cut | RED bars/session | ends flagged/session | wrong REDs/session |
|---|---|---|---|
| q0.90 | ~302 | ~231 | ~103 |
| q0.96 | ~123 | ~132 | ~9 |
| q0.98 | ~61 | ~70 | ~1 |

At q0.90 you'd see a hundred wrong REDs a day. At q0.98, about one. That's the difference between a lamp you learn to discount and one you believe.

You have three states, so use them as confidence levels rather than alert levels: **GREEN** below 0.000548 (half of bars, wrong 1 in 4300 — "alive, don't flinch"), **AMBER** from there to 0.402974 (q0.92), **RED** above 0.768366 (q0.96, right 92.5%). Two study params, both from this model's own distribution.

I'd start at q0.96 and keep q0.98 in your back pocket — if a 1-in-13 error rate still nags after a few sessions, tighten it. Live for a week, then decide; the number is a study param and costs nothing to change.

One live note: these percentages are from the holdout distribution. The threshold is a fixed number, so the fraction of RED bars will vary day to day — more on choppy days, fewer on trending ones. That's correct, not drift.

In [11]:
def ll(y, p):
    p = np.clip(np.asarray(p, np.float64), 1e-15, 1 - 1e-15)
    return float(-np.mean(y*np.log(p) + (1-y)*np.log(1-p)))

b = bars.merge(src[["timestamp", "cfbJMA"]], on="timestamp")
seg = (b.groupby(["date", "leg_id"])
         .agg(n_bars=("bar_index", "size"), cfb0=("cfbJMA", "first"),
              warm=("warm", "first"))
         .reset_index())
seg = seg[~seg.warm]
seg["q"] = pd.qcut(seg.cfb0, 4, labels=False)
print(seg.groupby("q").agg(n=("n_bars", "size"), median_len=("n_bars", "median"),
                           mean_len=("n_bars", "mean"), mean_cfb=("cfb0", "mean")))

       n  median_len   mean_len   mean_cfb
q                                         
0  80017        10.0  12.732319   7.233898
1  80017        10.0  13.003312   9.759948
2  80017        10.0  13.030843  12.686528
3  80017        10.0  12.939938  18.736392


In [14]:
b = bars.merge(src[["timestamp", "rawLast"]], on="timestamp")     # raw close in prod1
rows = []
for (d, lid), g in b[~b.warm].groupby(["date", "leg_id"]):
    if len(g) < 2: continue
    px = g["rawLast"].to_numpy(); dirn = g["jma_leg_dir"].iloc[0]
    ext = px.argmax() if dirn > 0 else px.argmin()
    rows.append((len(g), abs(px[ext] - px[0]), abs(px[-1] - px[0]), len(g) - 1 - ext))
s = pd.DataFrame(rows, columns=["bars", "pts_to_extreme", "pts_at_signal", "lag_bars"])
print(s.median(), "\nsegments/day:", len(s) / b.date.nunique())

bars              11.00
pts_to_extreme     2.75
pts_at_signal      3.25
lag_bars           5.00
dtype: float64 
segments/day: 271.32785467128025
